# 01 — Text Pre-processing Workbench

Runnable companion to **notes 03–08**: tokenization, stemming, lemmatization,
stopwords, POS tagging and named entity recognition.

Run the cells top to bottom. Every block is self-contained enough to re-run on its own
once setup is done.

| Section | Note |
|---|---|
| 1. Setup | — |
| 2. Tokenization | [03](../notes/03-Tokenization.md) |
| 3. Stemming | [04](../notes/04-Stemming.md) |
| 4. Lemmatization | [05](../notes/05-Lemmatization.md) |
| 5. Stopwords + full pipeline | [06](../notes/06-Stopwords-and-Cleaning-Pipeline.md) |
| 6. POS tagging | [07](../notes/07-POS-Tagging.md) |
| 7. Named entity recognition | [08](../notes/08-Named-Entity-Recognition.md) |
| 8. The reusable `clean_text` function | 06 |

## 1. Setup

Run once. The `_tab` / `_eng` duplicates cover the package renames in NLTK 3.8.2+, so the
same cell works on old and new versions.

In [ ]:
# !pip install nltk scikit-learn gensim pandas numpy

import nltk

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    nltk.download(pkg, quiet=True)

print("NLTK", nltk.__version__, "ready")

In [ ]:
corpus = """Hello Welcome, to Krish Naik's NLP tutorials.
Please do watch the entire course! to become expert in NLP."""

print(corpus)

## 2. Tokenization  ([note 03](../notes/03-Tokenization.md))

### 2.1 Sentence tokenization — paragraph → sentences

In [ ]:
from nltk.tokenize import sent_tokenize

documents = sent_tokenize(corpus)

print(type(documents), "of length", len(documents))
for i, s in enumerate(documents, 1):
    print(f"  [{i}] {s}")

**Three** sentences, not two — Punkt split on `!` as well as `.`

### Punkt is a trained model, not `text.split('.')`

In [ ]:
tricky = "Dr. Rao paid Rs. 5.5 lakh to Mr. Sharma. He was happy."

print("sent_tokenize :", sent_tokenize(tricky))
print("naive split   :", tricky.split("."))

Punkt knows `Dr.`, `Rs.` and `5.5` are not sentence ends. That is the whole reason to
use a library here.

### 2.2 Word tokenization

In [ ]:
from nltk.tokenize import word_tokenize

print(word_tokenize(corpus))

Note that `,` `.` `!` each became their own token, and `Naik's` split into `Naik` + `'s`.

### 2.3 The nested loop you will reuse everywhere

In [ ]:
for sentence in documents:
    print(word_tokenize(sentence))

### 2.4 All five tokenizers compared on the same input

In [ ]:
from nltk.tokenize import wordpunct_tokenize, TreebankWordTokenizer
from gensim.utils import simple_preprocess

text = "Hello Welcome, to Krish Naik's NLP tutorials in 2024!!"

print("word_tokenize       :", word_tokenize(text))
print()
print("wordpunct_tokenize  :", wordpunct_tokenize(text))
print()
print("TreebankWordTokenizer:", TreebankWordTokenizer().tokenize(text))
print()
print("simple_preprocess   :", simple_preprocess(text))

**What to notice**

| Tokenizer | `Naik's` | punctuation | final `.` | digits | case |
|---|---|---|---|---|---|
| `word_tokenize` | `Naik` + `'s` | separate | separate | kept | kept |
| `wordpunct_tokenize` | `Naik` + `'` + `s` | always separate | separate | kept | kept |
| `TreebankWordTokenizer` | `Naik` + `'s` | separate | **attached to previous word** | kept | kept |
| `simple_preprocess` | `naik` | **removed** | removed | **removed** | **lowered** |

`simple_preprocess` silently drops `2024`. That is either exactly what you want (Word2Vec
training) or a bug — know which.

In [ ]:
# The Treebank final-full-stop behaviour, isolated
print(word_tokenize("Hello world. Bye now."))
print(TreebankWordTokenizer().tokenize("Hello world. Bye now."))

## 3. Stemming  ([note 04](../notes/04-Stemming.md))

In [ ]:
words = ["eating", "eats", "eaten", "writing", "writes",
         "programming", "programs", "history", "finally", "finalized"]

### 3.1 Porter Stemmer

In [ ]:
from nltk.stem import PorterStemmer

porter = PorterStemmer()
for w in words:
    print(f"{w:14} ----> {porter.stem(w)}")

`eaten` unchanged and `history → histori`. Porter is a rule-based suffix stripper with
**no dictionary** — it never checks whether the output is a real word.

In [ ]:
for w in ['congratulations', 'sitting', 'goes', 'fairly', 'sportingly', 'happy', 'happier']:
    print(f"{w:16} ----> {porter.stem(w)}")

`happy → happi` and `happier → happi` shows *why* the terminal-`y` rule exists: it makes
those two unify. `history → histori` is the same rule misfiring.

### 3.2 RegexpStemmer — your own rule, and why `$` matters

In [ ]:
from nltk.stem import RegexpStemmer

reg = RegexpStemmer('ing$|s$|e$|able$', min=4)

print("eating     ->", reg.stem('eating'))
print("ingeating  ->", reg.stem('ingeating'), "   (leading 'ing' survives because of $)")

print()
print("no anchor  ->", RegexpStemmer('ing',  min=4).stem('ingeating'))
print("^ anchor   ->", RegexpStemmer('^ing', min=4).stem('ingeating'))

### 3.3 Snowball Stemmer — Porter improved

In [ ]:
from nltk.stem import SnowballStemmer

snowball = SnowballStemmer('english')
for w in words:
    print(f"{w:14} ----> {snowball.stem(w)}")

### 3.4 Where Snowball beats Porter

In [ ]:
for w in ['fairly', 'sportingly', 'goes', 'going', 'India']:
    print(f"{w:12}  porter={porter.stem(w):14} snowball={snowball.stem(w)}")

`fairly → fair` and `sportingly → sport` are the wins. `goes → goe` still fails for both —
only lemmatization gets that right.

Notice `India → india`: **Snowball lowercases its output**, which conveniently stops `India`
and `india` being two vocabulary entries.

In [ ]:
print(SnowballStemmer.languages)

## 4. Lemmatization  ([note 05](../notes/05-Lemmatization.md))

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
print(lemmatizer.lemmatize("going"))   # surprising!

Unchanged — because the **default is `pos='n'`** and *"a going"* is a valid noun.

### 4.1 The POS tag changes everything

In [ ]:
for tag in ['n', 'v', 'a', 'r']:
    print(tag, "->", lemmatizer.lemmatize("going", pos=tag))

### 4.2 The whole word list, noun vs verb vs Porter

In [ ]:
print(f"{'word':14} {'pos=n':14} {'pos=v':14} {'porter':14}")
print("-" * 58)
for w in words:
    print(f"{w:14} {lemmatizer.lemmatize(w):14} "
          f"{lemmatizer.lemmatize(w, pos='v'):14} {porter.stem(w):14}")

Read the `eaten` row (Porter fails, lemmatizer with `pos='v'` succeeds) and the `history`
row (Porter mangles it, lemmatizer leaves it alone).

In [ ]:
print(lemmatizer.lemmatize("goes",   pos='v'), "  (porter:", porter.stem("goes"), ")")
print(lemmatizer.lemmatize("better", pos='a'), " <- no stemmer can ever produce this")
print(lemmatizer.lemmatize("feet"))
print(lemmatizer.lemmatize("Krish"), lemmatizer.lemmatize("India"))

### 4.3 The production version: tag first, then lemmatize

In [ ]:
from nltk.corpus import wordnet

def wordnet_pos(treebank_tag):
    """Penn Treebank tag -> the 4 tags WordNet accepts."""
    if treebank_tag.startswith('J'): return wordnet.ADJ
    if treebank_tag.startswith('V'): return wordnet.VERB
    if treebank_tag.startswith('N'): return wordnet.NOUN
    if treebank_tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def smart_lemmatize(sentence):
    tagged = nltk.pos_tag(nltk.word_tokenize(sentence))
    return [lemmatizer.lemmatize(w, wordnet_pos(t)) for w, t in tagged]

s = "The striped bats are hanging on their feet and eating best"
print(nltk.word_tokenize(s))
print(smart_lemmatize(s))

`are → be`, `feet → foot`, `hanging → hang`, `bats → bat`. No stemmer comes close.

### 4.4 The speed cost

In [ ]:
import time

big = words * 2000        # 20,000 words

t = time.time(); _ = [snowball.stem(w) for w in big]
stem_time = time.time() - t

t = time.time(); _ = [lemmatizer.lemmatize(w, pos='v') for w in big]
lemm_time = time.time() - t

print(f"stemming      : {stem_time:.3f}s")
print(f"lemmatization : {lemm_time:.3f}s   ({lemm_time/stem_time:.1f}x slower)")

## 5. Stopwords & the full pipeline  ([note 06](../notes/06-Stopwords-and-Cleaning-Pipeline.md))

In [ ]:
from nltk.corpus import stopwords

sw = stopwords.words('english')
print(len(sw), "English stopwords")
print(sw[:25])

### 5.1 ⚠️ The trap: `not` is a stopword

In [ ]:
negations = [w for w in sw if w in ('no','nor','not') or "n't" in w]
print("negation words inside the default list:")
print(negations)

In [ ]:
STOP = set(stopwords.words('english'))

a = [w for w in "the food is good".split()     if w not in STOP]
b = [w for w in "the food is not good".split() if w not in STOP]

print("'the food is good'     ->", a)
print("'the food is not good' ->", b)
print("IDENTICAL?", a == b, " <- sentiment analysis is now impossible")

In [ ]:
keep = {'not','no','nor','never','none','cannot',
        "don't","doesn't","didn't","isn't","wasn't","aren't","weren't",
        "won't","wouldn't","couldn't","shouldn't","can't","hasn't","haven't",
        'but','however','although'}

MY_STOP = STOP - keep
print(f"default: {len(STOP)}   curated: {len(MY_STOP)}")

b2 = [w for w in "the food is not good".split() if w not in MY_STOP]
print("'the food is not good' ->", b2, " <- negation preserved")

### 5.2 The pipeline loop

In [ ]:
paragraph = """I have three visions for India. In 3000 years of our history,
people from all over the world have come and invaded us, captured our lands,
conquered our minds."""

sentences = nltk.sent_tokenize(paragraph)
print(sentences)

In [ ]:
def run_pipeline(paragraph, cleaner, lower=False):
    """tokenize -> drop stopwords -> apply `cleaner` -> rejoin."""
    out = nltk.sent_tokenize(paragraph)
    for i in range(len(out)):
        ws = nltk.word_tokenize(out[i])
        if lower:
            ws = [w.lower() for w in ws]
        ws = [cleaner(w) for w in ws if w.lower() not in STOP]
        out[i] = ' '.join(ws)
    return out

print("PORTER    :", run_pipeline(paragraph, porter.stem))
print()
print("SNOWBALL  :", run_pipeline(paragraph, snowball.stem))
print()
print("LEMMATIZER:", run_pipeline(paragraph, lambda w: lemmatizer.lemmatize(w, pos='v'), lower=True))

Compare the `history` token across the three: `histori`, `histori`, **`history`**.

### 5.3 Speed: hoisting the `set()` out of the loop

In [ ]:
sample = nltk.word_tokenize(paragraph) * 200

t = time.time()
_ = [w for w in sample if w not in set(stopwords.words('english'))]   # rebuilt every word
slow = time.time() - t

t = time.time()
_ = [w for w in sample if w not in STOP]                              # built once
fast = time.time() - t

print(f"set() inside loop : {slow:.3f}s")
print(f"set() hoisted out : {fast:.4f}s   ({slow/fast:.0f}x faster)")

## 6. POS tagging  ([note 07](../notes/07-POS-Tagging.md))

In [ ]:
tagged = nltk.pos_tag(nltk.word_tokenize("Taj Mahal is a beautiful monument"))
tagged

### 6.1 ⚠️ The mistake everyone makes: passing a string

In [ ]:
print("WRONG (string) :", nltk.pos_tag("Taj Mahal is beautiful")[:8])
print()
print("RIGHT (list)   :", nltk.pos_tag("Taj Mahal is beautiful".split()))

A string is iterable **by character**, so the tagger tags every letter. No exception is
raised — you just get nonsense. If you see single characters in the output, that is the cause.

### 6.2 The tagger is context-sensitive

In [ ]:
print(nltk.pos_tag(nltk.word_tokenize("I book a flight")))
print(nltk.pos_tag(nltk.word_tokenize("I read a book")))

Same string `book`, different tags (`VBP` vs `NN`). No lookup table could do this.

### 6.3 Looking up what a tag means

In [ ]:
for t in ['NNP', 'VBZ', 'JJ', 'PRP', 'CD']:
    nltk.help.upenn_tagset(t)

### 6.4 Using the tags as features

In [ ]:
tagged = nltk.pos_tag(nltk.word_tokenize(
    "The beautiful ancient Taj Mahal attracts millions of excited visitors"))

print("nouns      (what it's about):", [w for w, t in tagged if t.startswith('N')])
print("adjectives (the opinion)    :", [w for w, t in tagged if t.startswith('J')])
print("verbs                       :", [w for w, t in tagged if t.startswith('V')])

from collections import Counter
print("tag counts:", Counter(t for _, t in tagged).most_common())

## 7. Named entity recognition  ([note 08](../notes/08-Named-Entity-Recognition.md))

The three-step recipe: `word_tokenize` → `pos_tag` → `ne_chunk`.

In [ ]:
sentence = """The Eiffel Tower was built from 1887 to 1889 by French engineer
Gustave Eiffel, whose company specialized in building metal frameworks and structures."""

words_ner    = nltk.word_tokenize(sentence)
tag_elements = nltk.pos_tag(words_ner)
tree         = nltk.ne_chunk(tag_elements)

print(tree)

In [ ]:
# tree.draw()          # opens a Tk window — desktop only, NOT in Colab
from IPython.display import display
display(tree)            # renders inline in Jupyter

### 7.1 Turning the tree into a usable list

`hasattr(node, 'label')` is the idiom that separates entity subtrees from plain
`(word, tag)` tuples.

In [ ]:
def extract_entities(text):
    out = []
    for sent in nltk.sent_tokenize(text):
        t = nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(sent)))
        for node in t:
            if hasattr(node, 'label'):                 # a subtree -> an entity
                name = ' '.join(w for w, _ in node.leaves())
                out.append((name, node.label()))
    return out

for name, label in extract_entities(sentence):
    print(f"{label:15} {name}")

### 7.2 IOB format — what ML pipelines actually consume

In [ ]:
from nltk.chunk import tree2conlltags

for row in tree2conlltags(tree)[:10]:
    print(row)

`B-` = beginning of an entity, `I-` = inside the same entity, `O` = outside. The B/I
distinction is what separates "two adjacent PERSONs" from "one two-word PERSON".

### 7.3 NER makes mistakes — and misses money/dates

In [ ]:
text = """Apple Inc. was founded by Steve Jobs in Cupertino in 1976.
The company reported revenue of $394 billion in 2022."""

print("NLTK found:")
for name, label in extract_entities(text):
    print(f"  {label:15} {name}")

import re
print()
print("regex found money:", re.findall(r'\$\s?[\d,.]+\s?(?:billion|million|thousand)?', text))
print("regex found years:", re.findall(r'\b(?:19|20)\d{2}\b', text))

Production NER is usually a **hybrid**: a statistical model for PERSON/ORG/GPE plus
regexes for the highly-structured types (money, dates, emails, phone numbers, IDs).

## 8. The reusable `clean_text` function

This is what you carry into every project (notes 17 and 18 both use it).

In [ ]:
import re

STOP_KEEP_NEG = STOP - {'not','no','nor','never'}

def clean_text(text, mode='lemmatize', keep_negations=True, keep_digits=False):
    """Lowercase -> strip non-letters -> drop stopwords -> stem or lemmatize.

    mode          : 'lemmatize' | 'stem' | 'none'
    keep_negations: keep not/no/nor/never (essential for sentiment)
    keep_digits   : keep 0-9 (essential when numbers carry signal, e.g. spam)
    """
    stop    = STOP_KEEP_NEG if keep_negations else STOP
    pattern = '[^a-zA-Z0-9]' if keep_digits else '[^a-zA-Z]'

    text  = re.sub(pattern, ' ', str(text))     # NOTE: space, not '' -- see note 18
    text  = text.lower()
    words = text.split()
    words = [w for w in words if w not in stop]

    if   mode == 'lemmatize': words = [lemmatizer.lemmatize(w, pos='v') for w in words]
    elif mode == 'stem':      words = [porter.stem(w) for w in words]

    return ' '.join(words)


samples = [
    "Congratulations!!! You've WON a FREE ticket... claim now!!",
    "The food is NOT good at all.",
    "Call 87121 before 10am for your prize",
]

for s in samples:
    print(repr(s))
    print("  lemmatize   :", clean_text(s))
    print("  stem        :", clean_text(s, mode='stem'))
    print("  keep digits :", clean_text(s, keep_digits=True))
    print()

### The `''` vs `' '` bug, demonstrated

In [ ]:
bad  = re.sub('[^a-zA-Z]', '',  "great book,loved it")
good = re.sub('[^a-zA-Z]', ' ', "great book,loved it")

print("replace with '' :", repr(bad),  "<- words glued into a junk token")
print("replace with ' ':", repr(good), "<- correct")

---

## Exercises

1. Tokenize a paragraph of your own with all four word tokenizers and list three concrete
   differences.
2. Find five more words where Snowball beats Porter.
3. Run `clean_text` with `mode='stem'` and `mode='lemmatize'` on 100 reviews and compare the
   resulting vocabulary sizes (`len(set(all_tokens))`). Which is smaller, and why?
4. Extend `extract_entities` to also return MONEY and DATE using regexes.
5. Write `clean_text_spacy` using spaCy and compare speed on 5,000 documents.

**Next notebook:** [`02-text-to-vectors.ipynb`](02-text-to-vectors.ipynb) — turning this clean
text into numbers.